In [13]:
!pip install -q sentence-transformers faiss-cpu numpy

In [14]:
import os, re, json
import numpy as np
import faiss
from google.colab import drive, files
from sentence_transformers import SentenceTransformer

drive.mount('/content/drive')

DB_DIR     = "/content/drive/MyDrive/vector_db"
INDEX_PATH = os.path.join(DB_DIR, "faiss.index")
META_PATH  = os.path.join(DB_DIR, "chunks_metadata.json")
MODEL_NAME = "all-mpnet-base-v2"


FORCE_REBUILD = False

os.makedirs(DB_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]


def semantic_chunk(text, model, percentile=25, min_chars=150, max_chars=1000):


    sentences = split_sentences(text)
    if len(sentences) <= 1:
        return sentences

    emb = model.encode(sentences, normalize_embeddings=True, show_progress_bar=False)
    sims = np.sum(emb[:-1] * emb[1:], axis=1)
    threshold = np.percentile(sims, percentile)

    chunks = []
    current = [sentences[0]]
    current_len = len(sentences[0])

    for i in range(1, len(sentences)):
        sent = sentences[i]
        topic_changed = sims[i - 1] < threshold and current_len >= min_chars
        too_long = current_len + len(sent) + 1 > max_chars

        if topic_changed or too_long:
            chunks.append(" ".join(current))
            current = [sent]
            current_len = len(sent)
        else:
            current.append(sent)
            current_len += len(sent) + 1

    chunks.append(" ".join(current))
    return chunks

In [16]:
if os.path.exists(INDEX_PATH) and os.path.exists(META_PATH) and not FORCE_REBUILD:
    print("Vector DB already exists. Phase 1 skipped.")
    print("Go directly to Phase 2.")
else:
    print("Upload your 3 text files:")
    uploaded = files.upload()

    model = SentenceTransformer(MODEL_NAME)

    all_chunks = []
    metadata = []

    for file_name in uploaded.keys():
        with open(file_name, "r", encoding="utf-8") as f:
            text = f.read()

        chunks = semantic_chunk(text, model)
        print(f"{file_name}: {len(text)} chars -> {len(chunks)} chunks")

        for i, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            metadata.append({
                "source": file_name,
                "chunk_id": i,
                "text": chunk
            })


    embeddings = model.encode(all_chunks, normalize_embeddings=True, show_progress_bar=True)
    embeddings = np.array(embeddings, dtype="float32")


    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    faiss.write_index(index, INDEX_PATH)
    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump({"model": MODEL_NAME, "chunks": metadata}, f, ensure_ascii=False, indent=2)

    print("\nPhase 1 finished.")
    print("Total chunks stored:", index.ntotal)
    print("Saved to:", DB_DIR)

Vector DB already exists. Phase 1 skipped.
Go directly to Phase 2.


In [17]:
import os, json
import numpy as np
import faiss
from google.colab import drive
from sentence_transformers import SentenceTransformer

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

DB_DIR     = "/content/drive/MyDrive/vector_db"
INDEX_PATH = os.path.join(DB_DIR, "faiss.index")
META_PATH  = os.path.join(DB_DIR, "chunks_metadata.json")

assert os.path.exists(INDEX_PATH), "Vector DB not found. Run Phase 1 first."

index = faiss.read_index(INDEX_PATH)
with open(META_PATH, "r", encoding="utf-8") as f:
    db = json.load(f)

chunks_meta = db["chunks"]
model = SentenceTransformer(db["model"])

print("Vector DB loaded. Total chunks:", index.ntotal)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector DB loaded. Total chunks: 17


In [18]:
def search(query, top_k=3):

    q_vec = model.encode([query], normalize_embeddings=True)
    q_vec = np.array(q_vec, dtype="float32")

    scores, ids = index.search(q_vec, top_k)

    print("Question:", query)
    print("=" * 90)
    for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), start=1):
        item = chunks_meta[idx]
        print(f"\nAnswer {rank}")
        print(f"Reference (file name): {item['source']}  |  Chunk #{item['chunk_id']}")
        print(f"Cosine Similarity Score: {score:.4f}")
        print("-" * 90)
        print(item["text"])
    print("\n" + "=" * 90)


query = input("Enter your question: ")
search(query)

Enter your question: What is found in Africa?
Question: What is found in Africa?

Answer 1
Reference (file name): Animals.txt  |  Chunk #1
Cosine Similarity Score: 0.3532
------------------------------------------------------------------------------------------
The cheetah is the fastest land animal, able to run at about 110 kilometers per hour in short bursts. It lives mainly in the grasslands of Africa and uses its speed to catch gazelles. The giraffe is the tallest animal in the world, with some adults reaching about 5 meters in height. Its long neck helps it eat leaves from the tops of trees that other animals cannot reach.

Answer 2
Reference (file name): Animals.txt  |  Chunk #0
Cosine Similarity Score: 0.3424
------------------------------------------------------------------------------------------
Animals live in almost every part of the world, from hot deserts and dense forests to freezing polar regions and deep oceans. Scientists group them by what they eat, where they live, 